In [ ]:
!pip install transformers==4.44.2 -q
!pip install accelerate==0.34.2 -q

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # restrict to single GPU (avoids DataParallel bug)

import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0))

NumPy: 2.0.2
Torch: 2.10.0+cu128
Transformers: 4.44.2
CUDA available: True
Device count: 1
Device: Tesla T4


In [2]:
# Find the CSV path (Kaggle paths can vary)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [3]:
DATA_PATH = "/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv" # update if path differs above

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df = df[['review', 'label']]
df.head()

,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [5]:
SAMPLE_SIZE = 5000
df_small = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df_small['review'].tolist(),
    df_small['label'].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df_small['label']
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

print("Train label distribution:", np.bincount(train_labels))
print("Val label distribution:", np.bincount(val_labels))
print("Test label distribution:", np.bincount(test_labels))

Train size: 3500
Validation size: 750
Test size: 750
Train label distribution: [1737 1763]
Val label distribution: [372 378]
Test label distribution: [372 378]


In [6]:
model_name = "squeezebert/squeezebert-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model = model.to("cuda")

config.json:   0%|          | 0.00/500 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/103M [00:00<?, ?B/s]

Some weights of SqueezeBertForSequenceClassification were not initialized from the model checkpoint at squeezebert/squeezebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
MAX_LEN = 128

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
val_encodings   = tokenizer(val_texts,   truncation=True, padding=True, max_length=MAX_LEN)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=MAX_LEN)

In [8]:
class IMDBDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset   = IMDBDataset(val_encodings, val_labels)
test_dataset  = IMDBDataset(test_encodings, test_labels)

In [9]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [15]:
os.environ["TENSORBOARD_LOGGING_DIR"] = "./tb_logs"

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    max_grad_norm=1.0,
    bf16=True,
    fp16=False,
    report_to="tensorboard",
)

In [17]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.0
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

In [19]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.448400,0.440293,0.813333,0.822335
2,0.326400,0.422697,0.820000,0.825806
3,0.258800,0.435461,0.828000,0.830931
4,0.130800,0.496855,0.828000,0.836502
5,0.182200,0.519145,0.821333,0.822751


In [20]:
%load_ext tensorboard
%tensorboard --logdir ./tb_logs

<IPython.core.display.Javascript object>

In [21]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
print("Test metrics:", test_metrics)

Test metrics: {'eval_loss': 0.4588373303413391, 'eval_accuracy': 0.8186666666666667, 'eval_f1': 0.826530612244898, 'eval_runtime': 11.2738, 'eval_samples_per_second': 66.526, 'eval_steps_per_second': 2.129, 'epoch': 5.0}


In [22]:
model.save_pretrained("./squeezebert-imdb")
tokenizer.save_pretrained("./squeezebert-imdb")
print("Model and tokenizer saved.")

Model and tokenizer saved.


In [23]:
reloaded_model = AutoModelForSequenceClassification.from_pretrained("./squeezebert-imdb")
reloaded_tokenizer = AutoTokenizer.from_pretrained("./squeezebert-imdb")
reloaded_model = reloaded_model.to("cuda")
reloaded_model.eval()

print("Model reloaded successfully.")

Model reloaded successfully.


In [24]:
def predict_sentiment(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    return "positive" if pred == 1 else "negative"

sample_text = "The movie had stunning visuals but a very weak storyline overall."
result = predict_sentiment(sample_text, reloaded_model, reloaded_tokenizer)
print(f"Text: {sample_text}")
print(f"Predicted sentiment: {result}")

Text: The movie had stunning visuals but a very weak storyline overall.
Predicted sentiment: negative


In [25]:
samples = [
    "This is one of the best films I have ever seen, absolutely brilliant.",
    "Completely boring and a waste of two hours.",
    "The acting was great, but I wouldn't watch it again."
]

for text in samples:
    result = predict_sentiment(text, reloaded_model, reloaded_tokenizer)
    print(f"Text: {text}\nPredicted: {result}\n")

Text: This is one of the best films I have ever seen, absolutely brilliant.
Predicted: positive

Text: Completely boring and a waste of two hours.
Predicted: negative

Text: The acting was great, but I wouldn't watch it again.
Predicted: negative



In [26]:
import os
print(os.listdir("/kaggle/working"))

['squeezebert-imdb', 'results', '.virtual_documents']


In [27]:
import shutil
shutil.make_archive("/kaggle/working/squeezebert-imdb", 'zip', "/kaggle/working/squeezebert-imdb")
print("Zipped and ready to download.")

Zipped and ready to download.


In [29]:
import os
print(os.path.exists("/kaggle/working/squeezebert-imdb.zip"))
print(os.path.getsize("/kaggle/working/squeezebert-imdb.zip") / (1024*1024), "MB")

True
170.8653507232666 MB
